# Laboratorium 3 — Dane obrazowe: Drone vs Bird

**Cel:** utworzenie własnego mini-zbioru klas `bird` i `drone` (po 60 zdjęć), `ImageFolder`, `DataLoader`, dwóch potoków transformacji i wizualizacji.

**Dane:** [Kaggle: Drone vs Bird](https://www.kaggle.com/datasets/muhammadsaoodsarwar/drone-vs-bird). Notebook wykorzystuje **rzeczywiste zdjęcia**, bez generowania atrap. Pobieranie może wymagać internetu i autoryzacji Kaggle. Ten notebook nie przetwarza filmów — część dotycząca FPS/klatek jest nieadekwatna. Wyniki i pliki PNG powstają po wykonaniu komórek.

Najwygodniej uruchomić w Google Colab lub Kaggle Notebook. Jeśli pobieranie jest niedostępne, rozpakuj ręcznie pobrany zbiór do katalogu `drone-vs-bird` albo wskaż ścieżkę w `DATA_SOURCE`.

## 1. Biblioteki

In [ ]:
# Instaluj tylko brakujące biblioteki (np. w Colab):
# %pip install -q kagglehub torch torchvision pillow matplotlib
from pathlib import Path
from collections import Counter
import random, shutil, csv
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
OUTPUT = Path('obrazy_lab3')
OUTPUT.mkdir(exist_ok=True)
print('PyTorch:', torch.__version__)

## 2. Pobranie danych i odnalezienie klas

In [ ]:
DATA_SOURCE = None  # np. Path('/content/drone-vs-bird') dla ręcznie pobranego zbioru
if DATA_SOURCE is None:
    try:
        import kagglehub
        DATA_SOURCE = Path(kagglehub.dataset_download('muhammadsaoodsarwar/drone-vs-bird'))
    except Exception as exc:
        raise RuntimeError('Nie udało się pobrać Kaggle. Pobierz dane ręcznie i ustaw DATA_SOURCE na rozpakowany folder. Oryginalny błąd: ' + str(exc)) from exc
DATA_SOURCE = Path(DATA_SOURCE)
print('Źródło:', DATA_SOURCE)

EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
def category(path):
    # Oznaczenie klasy na podstawie katalogu lub części nazwy pliku.
    # Nie zgadujemy etykiet, jeśli klasy nie można jednoznacznie ustalić.
    pieces = [p.lower() for p in path.parts[-5:]]
    for piece in reversed(pieces[:-1]):
        if piece in ('bird', 'birds', 'bird images', 'bird_images') or piece.startswith('bird_'):
            return 'bird'
        if piece in ('drone', 'drones', 'drone images', 'drone_images') or piece.startswith('drone_'):
            return 'drone'
    stem = path.stem.lower()
    if stem.startswith(('bird_', 'bird-', 'bird ')):
        return 'bird'
    if stem.startswith(('drone_', 'drone-', 'drone ')):
        return 'drone'
    return None

all_paths = sorted((p for p in DATA_SOURCE.rglob('*') if p.is_file() and p.suffix.lower() in EXT), key=lambda p: str(p).lower())
files = {name: [p for p in all_paths if category(p) == name] for name in ['bird', 'drone']}
print('Liczby rozpoznanych plików:', {k: len(v) for k,v in files.items()})
if any(len(v) < 60 for v in files.values()):
    print('Przykładowe ścieżki:', [str(p.relative_to(DATA_SOURCE)) for p in all_paths[:25]])
    raise ValueError('Nie znaleziono co najmniej 60 obrazów każdej klasy. Sprawdź nazwy katalogów i dostosuj funkcję category() do struktury pobranych danych.')

## 3. Własny mini-zbiór w formacie ImageFolder

In [ ]:
ROOT = Path('dane_drone_bird')
N_PER_CLASS = 60
# Odtworzenie folderów umożliwia wielokrotne uruchamianie notebooka.
if ROOT.exists():
    shutil.rmtree(ROOT)
records = []
for name in ['bird', 'drone']:
    target = ROOT / name
    target.mkdir(parents=True)
    valid = []
    for path in files[name]:
        try:
            with Image.open(path) as im:
                im.verify()
            valid.append(path)
        except Exception:
            pass
    if len(valid) < N_PER_CLASS:
        raise ValueError(f'Za mało poprawnych obrazów klasy {name}: {len(valid)}')
    sampled = random.Random(SEED).sample(valid, N_PER_CLASS)
    for i, src in enumerate(sampled):
        dest = target / f'{name}_{i:03d}.jpg'
        with Image.open(src) as im:
            im = ImageOps.exif_transpose(im).convert('RGB')
            im.save(dest, quality=90)
        records.append((name, str(src), str(dest)))
with open('lista_obrazow.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['klasa', 'sciezka_zrodlowa', 'sciezka_w_minizbiorze'])
    writer.writerows(records)
print('Utworzono:', ROOT, '| obrazów:', len(records))

## 4. Transformacje ewaluacyjne i treningowe

In [ ]:
SIZE = (128, 128)
MEAN = (0.5, 0.5, 0.5)
STD = (0.5, 0.5, 0.5)
eval_tf = transforms.Compose([
    transforms.Resize(SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
train_tf = transforms.Compose([
    transforms.Resize(SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
# Jedna deterministyczna wersja do statystyk i jedna z augmentacją.
ds_eval = datasets.ImageFolder(ROOT, transform=eval_tf)
ds_train = datasets.ImageFolder(ROOT, transform=train_tf)
loader = DataLoader(ds_train, batch_size=12, shuffle=True, num_workers=0)
counts = Counter(ds_eval.targets)
print('Liczba klas:', len(ds_eval.classes))
print('class_to_idx:', ds_eval.class_to_idx)
print('Liczba obrazów na klasę:', {name: counts[idx] for name, idx in ds_eval.class_to_idx.items()})
print('Rozmiar po transformacji:', tuple(ds_eval[0][0].shape))
assert len(ds_eval.classes) == 2
assert all(counts[i] >= 30 for i in range(2))
assert ds_train.class_to_idx == ds_eval.class_to_idx

## 5. Siatka obrazów z jednej paczki

In [ ]:
def show_img(t):
    t = t.detach().cpu().permute(1, 2, 0).numpy()
    return np.clip(t * np.array(STD) + np.array(MEAN), 0, 1)

images, labels = next(iter(loader))
print('Kształt paczki:', tuple(images.shape), '| etykiety:', labels.tolist())
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for ax, image, label in zip(axes.flat, images, labels):
    ax.imshow(show_img(image))
    ax.set_title(ds_train.classes[label.item()])
    ax.axis('off')
fig.suptitle('Przykładowa paczka treningowa — Drone vs Bird')
fig.tight_layout()
fig.savefig(OUTPUT / '01_siatka_paczki.png', dpi=160)
plt.show()

## 6. Porównanie augmentacji jednego zdjęcia

In [ ]:
path, label_idx = ds_eval.samples[0]
with Image.open(path) as im:
    raw = im.convert('RGB')
fig, axes = plt.subplots(2, 4, figsize=(13, 7))
axes[0, 0].imshow(raw)
axes[0, 0].set_title('Oryginał')
axes[0, 1].imshow(show_img(eval_tf(raw)))
axes[0, 1].set_title('Bez augmentacji')
for i, ax in enumerate(list(axes.flat)[2:], start=1):
    ax.imshow(show_img(train_tf(raw)))
    ax.set_title(f'Augmentacja {i}')
for ax in axes.flat: ax.axis('off')
fig.suptitle(f'Augmentacja zdjęcia — klasa: {ds_eval.classes[label_idx]}')
fig.tight_layout()
fig.savefig(OUTPUT / '02_augmentacje.png', dpi=160)
plt.show()

## 7. Statystyki mini-zbioru i eksport wyników

In [ ]:
names = ds_eval.classes
values = [counts[ds_eval.class_to_idx[n]] for n in names]
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(names, values)
ax.set_ylabel('Liczba obrazów')
ax.set_title('Liczebność klas w mini-zbiorze')
for i, n in enumerate(values): ax.text(i, n + 0.4, str(n), ha='center')
ax.set_ylim(0, max(values) * 1.15)
fig.tight_layout()
fig.savefig(OUTPUT / '03_liczebnosc_klas.png', dpi=160)
plt.show()

report = f"""# Sprawozdanie — Laboratorium 3

## Temat
Dane obrazowe i wideo: potok i własny zbiór. Klasyfikacja obrazów ptaków i dronów.

## Cel
Przygotowanie dwuklasowego mini-zbioru fotografii, odczyt za pomocą ImageFolder i DataLoader, transformacje oraz podgląd paczki i augmentacji.

## Źródło danych
[Drone vs Bird — Muhammad Saood Sarwar, Kaggle](https://www.kaggle.com/datasets/muhammadsaoodsarwar/drone-vs-bird). Warunki licencji należy sprawdzić na stronie zbioru przed dalszą publikacją zdjęć. Zastosowano obrazy, nie ekstrakcję klatek z wideo.

## Przygotowanie danych i metoda
Utworzono strukturę `dane_drone_bird/bird/` i `dane_drone_bird/drone/`. W każdej klasie wylosowano po 60 poprawnych plików. Listę pochodzenia zapisano w `lista_obrazow.csv`.

Transformacja ewaluacyjna: Resize(128×128), ToTensor, Normalize(mean=0,5; std=0,5). Treningowa: to samo z dodatkowym RandomHorizontalFlip, RandomRotation(15) i ColorJitter. Losowe augmentacje stosowane są wyłącznie w wariancie treningowym.

## Wyniki
- Liczba klas: **{len(names)}**.
- Klasy: **{', '.join(names)}**.
- `class_to_idx`: **{ds_eval.class_to_idx}**.
- Obrazy: **{dict(zip(names, values))}**, razem **{sum(values)}**.
- Rozmiar po transformacji: **{tuple(ds_eval[0][0].shape)}** (C, H, W).
- Przykładowa paczka: **{tuple(images.shape)}** (N, C, H, W).

## Wizualizacje
![Siatka paczki](obrazy_lab3/01_siatka_paczki.png)

![Augmentacje](obrazy_lab3/02_augmentacje.png)

![Liczebność](obrazy_lab3/03_liczebnosc_klas.png)

## Wnioski
ImageFolder przypisał każdej klasie jednoznaczną etykietę liczbową, a DataLoader zwrócił tensory o zgodnym kształcie. Augmentacja daje różne warianty tego samego obrazu bez zmiany jego klasy. Mini-zbiór jest zbalansowany, ale ma ograniczoną wielkość i nie stanowi niezależnej oceny jakości klasyfikatora.

## Wideo, RODO i licencja
W tym ćwiczeniu **nie przetwarzano wideo**, więc FPS, indeksy klatek i BGR→RGB nie dotyczą tej realizacji. Obrazy są konwertowane do RGB przez Pillow. W przypadku widocznych i identyfikowalnych osób należy przed rozpowszechnianiem rozważyć podstawę przetwarzania i anonimizację. Przestrzegać warunków Kaggle oraz licencji autorów zdjęć; nie zakładać prawa do redystrybucji.

## Środowisko
Python, PyTorch, torchvision, Pillow, matplotlib, kagglehub. Seed: {SEED}.
"""
Path('sprawozdanie_lab3_wyniki.md').write_text(report, encoding='utf-8')
print('Zapisano sprawozdanie_lab3_wyniki.md oraz 3 wykresy w', OUTPUT)

## 8. Wnioski

Obowiązkowy zakres zadania jest zrealizowany dla **obrazów**, bez części wideo. Sieć MLP i eksperymenty dodatkowe są nieobowiązkowe i nie są tu wykonywane. Po wykonaniu notebooka zachowaj `sprawozdanie_lab3_wyniki.md`, `lista_obrazow.csv`, mini-zbiór oraz trzy pliki PNG z folderu `obrazy_lab3`.